# 04 — Phase 4: analysis (table + 3 plots, slide-ready)

Inputs: `<active-run>/measurements/metrics_ckpt{N}.json` (Phase 2),
`<active-run>/adaptation/ckpt-{N}/` (Phase 3), plus the active run's
`dashboard.jsonl` and `gsm8k_eval.jsonl` (Phase 1).

Outputs (→ slides / Research Doc):
- `outputs/results_table.csv` — checkpoint × {erank per layer, dormant per τ,
  GSM8K acc, SVAMP before/after/Δ}
- `outputs/fig_a_q_vs_updates.png` — Q vs training updates
- `outputs/fig_b_reward_vs_q.png` — reward curve with Q overlaid (twin axis —
  briefing-mandated motif; axes are color-matched to their series)
- `outputs/fig_c_scatter_q_vs_svamp.png` — Q(ckpt) vs fixed-budget SVAMP
  outcome, Spearman ρ
- `outputs/spearman_table.csv` — ρ for every Q variant vs every outcome

⚠️ n = 5 checkpoints: Spearman ρ here is a pilot-grade signal, not a
significance claim. |ρ| = 1 with n=5 still has p ≈ 0.017 at best.

CPU runtime is fine for this notebook.

In [ ]:
import json, sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/eaaj-pilot")
else:
    PROJECT_DIR = Path.cwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from src.repro import get_active_run

PILOT = json.loads(Path("pilot_config.json").read_text())
CKPTS = PILOT["stage_a"]["checkpoint_steps"]
LAYERS = PILOT["measurement"]["layers"]
RUN_DIR = get_active_run(PROJECT_DIR)
MEASURE_DIR = RUN_DIR / "measurements"
ADAPT_ROOT = RUN_DIR / "adaptation"
ANALYSIS_DIR = RUN_DIR / "analysis"
ANALYSIS_DIR.mkdir(exist_ok=True)

C = {"blue": "#2a78d6", "aqua": "#1baf7a", "yellow": "#eda100",
     "gray": "#52514e", "gray_light": "#b5b4ae"}
LAYER_COLORS = dict(zip(LAYERS, [C["blue"], C["aqua"], C["yellow"]]))
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300,
    "savefig.bbox": "tight", "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": "#e8e7e2",
    "grid.linewidth": 0.8, "axes.axisbelow": True, "font.size": 11})

## Assemble the master table

In [ ]:
rows = []
eval_log = [json.loads(l) for l in open(RUN_DIR / "gsm8k_eval.jsonl")]
gsm_acc = {r["step"]: r["accuracy"] for r in eval_log}

for n in CKPTS:
    m = json.loads((MEASURE_DIR / f"metrics_ckpt{n}.json").read_text())
    a = json.loads((ADAPT_ROOT / f"ckpt-{n}" / "summary.json").read_text())
    row = {"ckpt": n, "gsm8k_acc": gsm_acc.get(n, np.nan),
           "svamp_before": a["acc_before"], "svamp_after": a["acc_after"],
           "svamp_delta": a["delta_acc"]}
    for l in LAYERS:
        pl = m["per_layer"][f"layer{l}"]
        row[f"erank_L{l}"] = pl["erank"]
        row[f"erank_norm_L{l}"] = pl["erank_norm"]
        row[f"dormant025_L{l}"] = pl["dormant_frac_tau0.025"]
        row[f"dormant100_L{l}"] = pl["dormant_frac_tau0.1"]
        row[f"aniso_c_L{l}"] = pl["anisotropy_centered"]
    rows.append(row)

df = pd.DataFrame(rows).set_index("ckpt")
df.to_csv(ANALYSIS_DIR / "results_table.csv")
df.round(4)

## Plot (a) — Q vs training updates

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)

for l in LAYERS:
    ax1.plot(df.index, df[f"erank_L{l}"], "o-", lw=2, ms=7,
             color=LAYER_COLORS[l], label=f"layer {l}")
    ax1.annotate(f"L{l}", (df.index[-1], df[f"erank_L{l}"].iloc[-1]),
                 xytext=(6, 0), textcoords="offset points",
                 color=LAYER_COLORS[l], fontweight="bold", va="center")
ax1.set_ylabel("effective rank")
ax1.set_title("Q across GSM8K GRPO training (frozen 512-prompt probe)")
ax1.legend(frameon=False, loc="best")

for l in LAYERS:
    ax2.plot(df.index, df[f"dormant100_L{l}"], "o-", lw=2, ms=7,
             color=LAYER_COLORS[l], label=f"τ=0.1, layer {l}")
    ax2.plot(df.index, df[f"dormant025_L{l}"], "o--", lw=1.2, ms=4, alpha=0.6,
             color=LAYER_COLORS[l], label=f"τ=0.025, layer {l}")
ax2.set_ylabel("dormant fraction")
ax2.set_xlabel("GRPO updates")
ax2.set_xticks(CKPTS)
ax2.legend(frameon=False, ncol=3, fontsize=8)

fig.savefig(ANALYSIS_DIR / "fig_a_q_vs_updates.png")
plt.show()

## Plot (b) — dashboard flat / Q moving (reward + erank, twin axis)

In [ ]:
dash = pd.DataFrame([json.loads(l) for l in open(RUN_DIR / "dashboard.jsonl")])
reward_col = next(c for c in dash.columns if c == "reward" or c.endswith("/reward")
                  or c.endswith("reward_mean") or c == "rewards/exact_answer_reward/mean")
dash = dash.dropna(subset=[reward_col])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(dash["step"], dash[reward_col], color=C["gray_light"], lw=1, alpha=0.8)
roll = dash[reward_col].rolling(10, min_periods=1).mean()
ax.plot(dash["step"], roll, color=C["gray"], lw=2.2, label="reward (10-step mean)")
ax.set_xlabel("GRPO updates")
ax.set_ylabel("mean reward (dashboard)", color=C["gray"])
ax.tick_params(axis="y", labelcolor=C["gray"])
ax.grid(False)

axq = ax.twinx()  # briefing-mandated twin-axis motif
axq.plot(df.index, df["erank_L12"], "o-", color=C["blue"], lw=2.5, ms=8,
         label="effective rank (layer 12)")
axq.set_ylabel("effective rank, layer 12 (Q)", color=C["blue"])
axq.tick_params(axis="y", labelcolor=C["blue"])
axq.spines.right.set_visible(True)
axq.spines.right.set_color(C["blue"])

ax.set_title("Dashboard vs Q: reward curve with effective rank overlaid")
fig.legend(loc="lower left", bbox_to_anchor=(0.12, 0.14), frameon=False)
fig.savefig(ANALYSIS_DIR / "fig_b_reward_vs_q.png")
plt.show()

## Plot (c) — Q vs fixed-budget SVAMP outcome (Spearman ρ)

In [ ]:
q_col, out_col = "erank_L12", "svamp_delta"
rho, p = spearmanr(df[q_col], df[out_col])

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(df[q_col], df[out_col], s=90, color=C["blue"], zorder=3)
for n in df.index:
    ax.annotate(f"ckpt {n}", (df.loc[n, q_col], df.loc[n, out_col]),
                xytext=(8, 4), textcoords="offset points", fontsize=9,
                color="#0b0b0b")
ax.set_xlabel("effective rank @ layer 12 (Q at checkpoint)")
ax.set_ylabel("SVAMP Δaccuracy (fixed 50-update budget)")
ax.set_title(f"RQ1 pilot: Spearman ρ = {rho:.2f} (p = {p:.3f}, n = 5)")
fig.savefig(ANALYSIS_DIR / "fig_c_scatter_q_vs_svamp.png")
plt.show()

## Spearman table — every Q variant vs every outcome

In [ ]:
q_vars = [c for c in df.columns if c.startswith(("erank", "dormant", "aniso"))]
outcomes = ["svamp_delta", "svamp_after", "gsm8k_acc"]
tab = []
for q in q_vars:
    row = {"q_variant": q}
    for o in outcomes:
        row[f"rho_{o}"] = spearmanr(df[q], df[o])[0]
    tab.append(row)
spear = pd.DataFrame(tab).set_index("q_variant").round(3)
spear.to_csv(ANALYSIS_DIR / "spearman_table.csv")
print(spear.to_string())
print("\nInterpretation guide (briefing §7): GOOD = erank declines with updates "
      "AND correlates positively with SVAMP Δacc. n=5 — treat as pilot signal.")